In [0]:
display(dbutils.fs.ls("abfss://azure-data@azuredataengine.dfs.core.windows.net/"))

In [0]:
display(dbutils.fs.ls("abfss://azure-data@azuredataengine.dfs.core.windows.net/Raw/"))

In [0]:
raw_file = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Raw/data_00e410b6-9821-43f6-8721-0f63e3840c34_07011ba1-6d68-4922-ba05-6ee977820edf.json"

df = spark.read.option("multiline", "true").json(raw_file)

display(df)

In [0]:
raw_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Raw/"

ingested_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Ingested/"

df_raw = (
    spark.read
    .option("multiline", "true")
    .json(raw_path)
)

display(df_raw)

In [0]:
df_raw.write \
    .mode("overwrite") \
    .json(ingested_path)

In [0]:
display(dbutils.fs.ls("abfss://azure-data@azuredataengine.dfs.core.windows.net/Ingested/"))

In [0]:
ingested_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Ingested/"

df_ingested = (
    spark.read
    .option("multiline", "true")
    .json(ingested_path)
)

display(df_ingested)

In [0]:
from pyspark.sql.functions import explode, col

df_races = df_ingested.select(
    explode(col("RaceTable.Races")).alias("race")
)

display(df_races)

In [0]:
display(df_ingested.select("MRData"))

In [0]:
from pyspark.sql.functions import explode, col

df_races = df_ingested.select(
    explode(col("MRData.RaceTable.Races")).alias("race")
)

display(df_races)

In [0]:
df_races_flat = df_races.select(
    col("race.season").alias("season"),
    col("race.round").alias("round"),
    col("race.raceName").alias("race_name"),
    col("race.date").alias("race_date"),
    col("race.Circuit.circuitId").alias("circuit_id"),
    col("race.Circuit.circuitName").alias("circuit_name"),
    col("race.Circuit.Location.locality").alias("locality"),
    col("race.Circuit.Location.country").alias("country"),
    col("race.Circuit.Location.lat").alias("latitude"),
    col("race.Circuit.Location.long").alias("longitude")
)

display(df_races_flat)

In [0]:
presentation_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Presentation/"

df_races_flat.write \
    .mode("overwrite") \
    .parquet(presentation_path)

In [0]:
display(dbutils.fs.ls("abfss://azure-data@azuredataengine.dfs.core.windows.net/Presentation/"))

In [0]:
df_presentation = spark.read.parquet(presentation_path)

display(df_presentation)

In [0]:
from pyspark.sql.functions import count

df_races_per_season = (
    df_presentation
    .groupBy("season")
    .agg(count("*").alias("race_count"))
    .orderBy("season")
)

display(df_races_per_season)

In [0]:
df_presentation.select(
    "season",
    "round",
    "race_name"
).distinct().count()

In [0]:
df_clean = df_presentation.dropDuplicates([
    "season",
    "round",
    "race_name"
])

display(df_clean)

In [0]:
df_races_per_season = (
    df_clean
    .groupBy("season")
    .count()
    .withColumnRenamed("count", "race_count")
    .orderBy("season")
)

display(df_races_per_season)

In [0]:
analyze_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Presentation/races_per_season/"

df_races_per_season.write \
    .mode("overwrite") \
    .parquet(analyze_path)

In [0]:
display(dbutils.fs.ls(analyze_path))

In [0]:
df_analyze = spark.read.parquet(analyze_path)

display(df_analyze)

In [0]:
df_races_by_country = (
    df_clean
    .groupBy("country")
    .count()
    .withColumnRenamed("count", "race_count")
    .orderBy("country")
)

display(df_races_by_country)

In [0]:
country_analysis_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Presentation/races_by_country/"

df_races_by_country.write \
    .mode("overwrite") \
    .parquet(country_analysis_path)

In [0]:
display(dbutils.fs.ls(country_analysis_path))

In [0]:
df_country_analysis = spark.read.parquet(country_analysis_path)

display(df_country_analysis)